<a href="https://colab.research.google.com/github/finntangvorakasem/urap2025/blob/main/TANGVORAKASEM_FINN_LAB8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 8 Neural Language Model
A language model predicts the next word in the sequence based on the specific words that have come before it in the sequence.

It is also possible to develop language models at the character level using neural networks. The benefit of character-based language models is their small vocabulary and flexibility in handling any words, punctuation, and other document structure. This comes at the cost of requiring larger models that are slower to train.

Nevertheless, in the field of neural language models, character-based models offer a lot of promise for a general, flexible and powerful approach to language modeling.

As a prerequisite for the lab, make sure to pip install:
- keras
- tensorflow
- h5py

# Source Text Creation

To start out with, we'll be using a simple nursery rhyme. It's quite short so we can actually train something on your CPU and see relatively interesting results. Please copy and paste the following text in a text file and save it as "rhymes.txt". Place this in the same directory as this jupyter notebook:

In [ ]:
!pip install tensorflow
!pip install keras
!pip install h5py

In [ ]:

s='Sing a song of sixpence,\
A pocket full of rye.\
Four and twenty blackbirds,\
Baked in a pie.\
When the pie was opened\
The birds began to sing;\
Wasn’t that a dainty dish,\
To set before the king.\
The king was in his counting house,\
Counting out his money;\
The queen was in the parlour,\
Eating bread and honey.\
The maid was in the garden,\
Hanging out the clothes,\
When down came a blackbird\
And pecked off her nose.'

with open('rhymes.txt','w') as f:
  f.write(s)

    Sing a song of sixpence,
    A pocket full of rye.
    Four and twenty blackbirds,
    Baked in a pie.

    When the pie was opened
    The birds began to sing;
    Wasn’t that a dainty dish,
    To set before the king.

    The king was in his counting house,
    Counting out his money;
    The queen was in the parlour,
    Eating bread and honey.

    The maid was in the garden,
    Hanging out the clothes,
    When down came a blackbird
    And pecked off her nose.

# Sequence Generation

A language model must be trained on the text, and in the case of a character-based language model, the input and output sequences must be characters.

The number of characters used as input will also define the number of characters that will need to be provided to the model in order to elicit the first predicted character.

After the first character has been generated, it can be appended to the input sequence and used as input for the model to generate the next character.

Longer sequences offer more context for the model to learn what character to output next but take longer to train and impose more burden on seeding the model when generating text.

We will use an arbitrary length of 10 characters for this model.

There is not a lot of text, and 10 characters is a few words.

We can now transform the raw text into a form that our model can learn; specifically, input and output sequences of characters.

In [ ]:
#load doc into memory
def load_doc(filename):
    # open the file as read only
    file = open(filename, 'r')
    # read all text
    text = file.read()
    # close the file
    file.close()
    return text
# save tokens to file, one dialog per line
def save_doc(lines, filename):
    data = '\n'.join(lines)
    file = open(filename, 'w')
    file.write(data)
    file.close()

In [ ]:
#load text
raw_text = load_doc('rhymes.txt')
print(raw_text)

# clean
tokens = raw_text.split()
raw_text = ' '.join(tokens)

# organize into sequences of characters
length = 20
sequences = list()
for i in range(length, len(raw_text)):
    # select sequence of tokens
    seq = raw_text[i-length:i+1]
    # store
    sequences.append(seq)
print('Total Sequences: %d' % len(sequences))

Sing a song of sixpence,A pocket full of rye.Four and twenty blackbirds,Baked in a pie.When the pie was openedThe birds began to sing;Wasn’t that a dainty dish,To set before the king.The king was in his counting house,Counting out his money;The queen was in the parlour,Eating bread and honey.The maid was in the garden,Hanging out the clothes,When down came a blackbirdAnd pecked off her nose.
Total Sequences: 374


In [ ]:
# save sequences to file
out_filename = 'char_sequences.txt'
save_doc(sequences, out_filename)

# Train a Model
In this section, we will develop a neural language model for the prepared sequence data.

The model will read encoded characters and predict the next character in the sequence. A Long Short-Term Memory recurrent neural network hidden layer will be used to learn the context from the input sequence in order to make the predictions.

In [ ]:
from numpy import array
from pickle import dump
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout

# load doc into memory
def load_doc(filename):
    # open the file as read only
    file = open(filename, 'r')
    # read all text
    text = file.read()
    # close the file
    file.close()
    return text

In [ ]:
# load

in_filename = 'char_sequences.txt'
raw_text = load_doc(in_filename)
lines = raw_text.split('\n')

The sequences of characters must be encoded as integers.This means that each unique character will be assigned a specific integer value and each sequence of characters will be encoded as a sequence of integers. We can create the mapping given a sorted set of unique characters in the raw input data. The mapping is a dictionary of character values to integer values.

Next, we can process each sequence of characters one at a time and use the dictionary mapping to look up the integer value for each character. The result is a list of integer lists.

We need to know the size of the vocabulary later. We can retrieve this as the size of the dictionary mapping.

In [ ]:
# integer encode sequences of characters
chars = sorted(list(set(raw_text)))
mapping = dict((c, i) for i, c in enumerate(chars))
sequences = list()
for line in lines:
    # integer encode line
    encoded_seq = [mapping[char] for char in line]
    # store
    sequences.append(encoded_seq)

# vocabulary size
vocab_size = len(mapping)
print('Vocabulary Size: %d' % vocab_size)

# separate into input and output
sequences = array(sequences)
X, y = sequences[:,:-1], sequences[:,-1]
sequences = [to_categorical(x, num_classes=vocab_size) for x in X]
X = array(sequences)
y = to_categorical(y, num_classes=vocab_size)

Vocabulary Size: 38


The model is defined with an input layer that takes sequences that have 10 time steps and 38 features for the one hot encoded input sequences. Rather than specify these numbers, we use the second and third dimensions on the X input data. This is so that if we change the length of the sequences or size of the vocabulary, we do not need to change the model definition.

The model has a single LSTM hidden layer with 75 memory cells. The model has a fully connected output layer that outputs one vector with a probability distribution across all characters in the vocabulary. A softmax activation function is used on the output layer to ensure the output has the properties of a probability distribution.

The model is learning a multi-class classification problem, therefore we use the categorical log loss intended for this type of problem. The efficient Adam implementation of gradient descent is used to optimize the model and accuracy is reported at the end of each batch update. The model is fit for 50 training epochs.

# To Do:
- Try different numbers of memory cells
- Try different types and amounts of recurrent and fully connected layers
- Try different lengths of training epochs
- Try different sequence lengths and pre-processing of data
- Try regularization techniques such as Dropout

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = Sequential()
model.add(LSTM(50, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dense(vocab_size, activation='softmax'))
print(model.summary())

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    verbose=1
)

Model: "sequential_34"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_37 (LSTM)                  │ (None, 50)             │        17,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 38)             │         1,938 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,738 (77.10 KB)

 Trainable params: 19,738 (77.10 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - accuracy: 0.0975 - loss: 3.6262 - val_accuracy: 0.0533 - val_loss: 3.6100
Epoch 2/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.1598 - loss: 3.5670 - val_accuracy: 0.0533 - val_loss: 3.5384
Epoch 3/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.1851 - loss: 3.4194 - val_accuracy: 0.1200 - val_loss: 3.3631
Epoch 4/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1673 - loss: 3.1488 - val_accuracy: 0.1333 - val_loss: 3.3658
Epoch 5/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1643 - loss: 3.1059 - val_accuracy: 0.1333 - val_loss: 3.3292
Epoch 6/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1269 - loss: 3.1410 - val_accuracy: 0.1333 - val_loss: 3.3142
Epoch 7/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1482 - loss: 3.0753 - val_accuracy: 0.1333 - val_loss: 3.3184
Epoch 8/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1528 - loss: 3.0320 - val_accurac

In [ ]:
# define model
#model = Sequential()
#model.add(LSTM(75, input_shape=(X.shape[1], X.shape[2])))
#model.add(Dense(vocab_size, activation='softmax'))
#print(model.summary())
# compile model
#model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# fit model
#history=model.fit(X, y, epochs=200)

In [ ]:
# save the model to file
model.save('model.h5')
# save the mapping
dump(mapping, open('mapping.pkl', 'wb'))

# Generating Text

We must provide sequences of 10 characters as input to the model in order to start the generation process. We will pick these manually. A given input sequence will need to be prepared in the same way as preparing the training data for the model.

In [ ]:
from pickle import load
import numpy as np
from keras.models import load_model
from tensorflow.keras.utils import to_categorical
from keras.preprocessing.sequence import pad_sequences
import os # Import the os module
import h5py

# generate a sequence of characters with a language model
def generate_seq(model, mapping, seq_length, seed_text, n_chars):
    in_text = seed_text
    # generate a fixed number of characters
    for _ in range(n_chars):
        # encode the characters as integers
        encoded = [mapping[char] for char in in_text]
        # truncate sequences to a fixed length
        encoded = pad_sequences([encoded], maxlen=seq_length, truncating='pre')
        # one hot encode
        encoded = to_categorical(encoded, num_classes=len(mapping))
        # predict character
        yhat = np.argmax(model.predict(encoded), axis=-1)
        # reverse map integer to character
        out_char = ''
        for char, index in mapping.items():
            if index == yhat:
                out_char = char
                break
        # append to input
        in_text += char
    return in_text

# Get the absolute path of the current directory.
current_dir = os.getcwd()
# Print the current directory to check where the notebook is looking for files.
print(f"Current directory: {current_dir}")
# Define the expected path to the model file. You might need to modify this.
model_path = os.path.join(current_dir, 'model.h5')
# Load the model using the defined path
model = load_model(model_path)
# load the mapping
mapping = load(open('mapping.pkl', 'rb'))

Current directory: /content


Running the example generates three sequences of text.

The first is a test to see how the model does at starting from the beginning of the rhyme. The second is a test to see how well it does at beginning in the middle of a line. The final example is a test to see how well it does with a sequence of characters never seen before.

In [ ]:
# test start of rhyme
print(generate_seq(model, mapping, 10, 'Sing a son', 20))
# test mid-line
print(generate_seq(model, mapping, 10, 'king was i', 20))
# test not in original
print(generate_seq(model, mapping, 10, 'hello worl', 20))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Sing a soneWhe  af ootnin brar
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━

If the results aren't satisfactory, try out the suggestions above or these below:
- Padding. Update the example to provides sequences line by line only and use padding to fill out each sequence to the maximum line length.
- Sequence Length. Experiment with different sequence lengths and see how they impact the behavior of the model.
- Tune Model. Experiment with different model configurations, such as the number of memory cells and epochs, and try to develop a better model for fewer resources.


# Deliverables to receive credit

1. (4 points) Optimize the cells above to tune the model so that it generates text that closely resembles the orginal line from the rhyme, or at least generates sensible words. It's okay if the third example using unseen text still looks somewhat strange  though. Again, this is a toy problem, as language models require a lot of computation. This toy problem is great for rapid experimentation to explore different aspects of deep learning language models.
2. (3 points) Write a function to split the text corpus file into training and validation and pipe the validation data into the model.fit() function to be able to track validation error per epoch. Lookup Keras documentation to see how this is handled.
3. (3 points) Write a summary (methods and results) in the cells below of the different things you applied. You must include your intuitions behind what did work and what did not work well.
4. (Extra Credit 2.5 points) Do something even more interesting. Try a different source text. Train a word-level model. We'll leave it up to your creativity to explore and write a summary of your methods and results.


In [ ]:
##Part 2 (3 points) and Part 1 are done above, as the instructions say to optimize the cell above. I was unsure but the "above" instruction prompted me to edit above.

In [ ]:
#baseline
model = Sequential()
model.add(LSTM(75, input_shape=(X.shape[1], X.shape[2])))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history1 = model.fit(X, y, epochs=100, verbose=0)
print(history1.history['accuracy'][-1])
print(history1.history['loss'][-1])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


0.9919785857200623
0.3228733539581299


In [ ]:
#more LSTM cells
model = Sequential()
model.add(LSTM(150, input_shape=(X.shape[1], X.shape[2])))  #changed to 150
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history2 = model.fit(X, y, epochs=100, verbose=0)
print(history2.history['accuracy'][-1])
print(history2.history['loss'][-1])

1.0
0.02903035655617714


In [ ]:
#less LSTM cells
model = Sequential()
model.add(LSTM(50, input_shape=(X.shape[1], X.shape[2])))  #changed to 50
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history2 = model.fit(X, y, epochs=100, verbose=0)
print(history2.history['accuracy'][-1])
print(history2.history['loss'][-1])
#not the best

0.8021390438079834
0.9891538619995117


In [ ]:
#add dropout
model = Sequential()
model.add(LSTM(75, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.3))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history3 = model.fit(X, y, epochs=100, verbose=0)
print(history3.history['accuracy'][-1])
print(history3.history['loss'][-1])

0.7887700796127319
0.8999008536338806


In [ ]:
#2 layers + 200 epochs
model = Sequential()
model.add(LSTM(100, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))  #First LSTM
model.add(LSTM(50)) #Second
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history5 = model.fit(X, y, epochs=200, verbose=0)
print(history5.history['accuracy'][-1])
print(history5.history['loss'][-1])
print(model.summary())

1.0
0.06525711715221405


Model: "sequential_39"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_42 (LSTM)                  │ (None, 20, 100)        │        55,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_43 (LSTM)                  │ (None, 50)             │        30,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 38)             │         1,938 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 263,216 (1.00 MB)

 Trainable params: 87,738 (342.73 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 175,478 (685.46 KB)

None


In [ ]:
#Longer Sequence Length
length = 20
sequences = list()
for i in range(length, len(raw_text)):
    seq = raw_text[i-length:i+1]
    sequences.append(seq)

#same code
chars = sorted(list(set(raw_text)))
mapping = dict((c, i) for i, c in enumerate(chars))
encoded_sequences = []
for line in sequences:
    encoded_seq = [mapping[char] for char in line]
    encoded_sequences.append(encoded_seq)

vocab_size = len(mapping)
sequences_array = array(encoded_sequences)
X, y = sequences_array[:,:-1], sequences_array[:,-1]
sequences_encoded = [to_categorical(x, num_classes=vocab_size) for x in X]
X = array(sequences_encoded)
y = to_categorical(y, num_classes=vocab_size)

model = Sequential()
model.add(LSTM(75, input_shape=(X.shape[1], X.shape[2])))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history6 = model.fit(X, y, epochs=200, verbose=0)
print(history6.history['accuracy'][-1])
print(history6.history['loss'][-1])
#best so far

In [ ]:
print("20 length with 100 epochs: accuracy 0.999563992023468, loss 0.0026755130384117365 ")
print("20 length with 200 epochs: accuracy 0.999127984046936, loss 0.0018064897740259767 ")


20 length with 100 epochs: accuracy 0.999563992023468, loss 0.0026755130384117365 
20 length with 200 epochs: accuracy 0.999127984046936, loss 0.0018064897740259767 


In [ ]:
print("""Key Observations and Intuitions

1. Extreme Overfitting Throughout: All configurations exhibited severe overfitting,
with training accuracy consistently 70-90% higher than validation accuracy

2. Model Complexity vs Performance:
   - Reducing LSTM cells from 75 to 50 slightly improved validation accuracy (4% to 10-20%)
   - However, the overfitting gap remained substantial

3. Training Progression:
   - Validation accuracy peaked around epochs 130-145 at ~20%
   - Continued training beyond this point showed no improvement
   - Training accuracy continued climbing to near-perfect levels

4. Validation Loss Trajectory:
   - Validation loss increased throughout training (3.6 to 4.5)
   - This divergence from training loss shows the model was memorizing rather than learning patterns

What Worked (Initially)
1. Longer Sequence Length (20 characters):
   - Provided more context for predictions compared to the original 10-character sequences
   - Allowed the model to capture longer character dependencies
   - This was the most impactful improvement from the baseline (the rest weren’t impactful and were negligible)
   - Prior to training/test split, this sequence and the baseline settings (LSTM 75 cells, 100 epochs),
   resulted in the highest accuracy, while the one with 200 epochs has the lowest validation loss.
   But this proved to be misleading as validation accuracy is measured.

2. Reduced Model Capacity (50 LSTM cells):
   - Marginally reduced overfitting compared to 75 cells
   - Validation accuracy improved from ~4% to ~10-20%
   - Demonstrated that simpler models can sometimes generalize better

3. Train/Validation Split:
   - Reveals severe overfitting (1.00 accuracy and 0.12 final validation accuracy)
   - Without this split, the near-perfect training accuracy initially achieved would have been misleading

What Didn't Work

1. Dropout:
   - Initially attempted Dropout(0.3-0.5 to 0.7-0.9)
   - Result: Validation accuracy actually decreased, down to 3-4% in some cases.
   - Intuition: With only ~300 training samples, dropout was too aggressive.
   The model needed every connection available to learn even basic patterns.

2. Additional Training Epochs (from 150 to 300):
   - Training beyond 200 epochs showed no validation improvement
   - Validation loss continued to increase while training loss decreased
   - Intuition: The model had already learned all generalizable patterns from the limited data;
   additional training only increased memorization

3. Increased Model Capacity (tested but not shown in final):
   - Larger models (75+ LSTM cells) performed worse on validation
   - Intuition: More parameters require more data to train effectively. Our tiny corpus couldn't support larger architectures.

---

Intuitions and Analysis

1. Dataset Size:The model simply didn't have enough diverse examples to learn general character-level language patterns

2. Repetition: The nursery rhyme contains highly repetitive patterns ("The king was in...", "The queen was in...")
that the model memorized as exact sequences rather than learning flexible rules. This experiment revealed that for
extremely small datasets, it is challenging to overcome the lack of data**.

- Simpler models (50 cells) underfit - they lack capacity to learn complex patterns
- Complex models (75+ cells) overfit - they memorize training data
- The "sweet spot" still exhibits 70%+ overfitting because the dataset is too small

Overall, I think that a bigger data would've been monumental to enhance accuracy, as
it will allow the model to generalize well, not merely memorizing from small samples, as well as
potentially using word-level prediction instead of character.
""")

Key Observations and Intuitions

1. Extreme Overfitting Throughout: All configurations exhibited severe overfitting, 
with training accuracy consistently 70-90% higher than validation accuracy

2. Model Complexity vs Performance:
   - Reducing LSTM cells from 75 to 50 slightly improved validation accuracy (4% to 10-20%)
   - However, the overfitting gap remained substantial

3. Training Progression: 
   - Validation accuracy peaked around epochs 130-145 at ~20%
   - Continued training beyond this point showed no improvement
   - Training accuracy continued climbing to near-perfect levels

4. Validation Loss Trajectory: 
   - Validation loss increased throughout training (3.6 to 4.5)
   - This divergence from training loss shows the model was memorizing rather than learning patterns

What Worked (Initially)
1. Longer Sequence Length (20 characters):
   - Provided more context for predictions compared to the original 10-character sequences
   - Allowed the model to capture longer charact